In [1]:
# SCD2 Slowly Changes Dimension

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,when,lit,current_date

In [3]:
spark = SparkSession.builder.appName("SCD1").getOrCreate()

In [4]:
schema = StructType(
    [
        StructField("Customer_id",IntegerType(),True),
        StructField("Name",StringType(),True),
        StructField("Location",StringType(),True),
        StructField("StartDate",StringType(),True),
        StructField("EndDate",StringType(),True),
        StructField("Status",StringType(),True),
    ]
)

Target_data = [
    (1001, "Argha", "Kolkata", "2023-01-01", None, "Y"),
    (1002, "Rahul", "Delhi", "2023-01-01", None, "Y"),
    (1003, "John", "Mumbai", "2023-01-05", None, "Y"),
    (1004, "Sham", "Chennai", "2023-01-08", None, "Y"),
    (1005, "Champ", "Noida", "2022-05-04", "2025-03-02", "N")
]

define_schema = StructType([
    StructField("Customer_id",IntegerType(),True),
    StructField("Name",StringType(),True),
    StructField("Location",StringType(),True)
])

source_data = [
    (1001, "Argha", "Mumbai"),  # changed city
    (1002, "Rahul", "Delhi"),   # no change
    (1003, "John", "pune"),     # changed city
    (1006, "Amit", "Pune")      # new record
]

In [5]:
dim_df = spark.createDataFrame(Target_data,schema)
src_df = spark.createDataFrame(source_data,define_schema)
dim_df.show()
src_df.show()

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1001|Argha| Kolkata|2023-01-01|      null|     Y|
|       1002|Rahul|   Delhi|2023-01-01|      null|     Y|
|       1003| John|  Mumbai|2023-01-05|      null|     Y|
|       1004| Sham| Chennai|2023-01-08|      null|     Y|
|       1005|Champ|   Noida|2022-05-04|2025-03-02|     N|
+-----------+-----+--------+----------+----------+------+

+-----------+-----+--------+
|Customer_id| Name|Location|
+-----------+-----+--------+
|       1001|Argha|  Mumbai|
|       1002|Rahul|   Delhi|
|       1003| John|    pune|
|       1006| Amit|    Pune|
+-----------+-----+--------+



In [6]:
updated_dim_df = dim_df.filter(col('Status')=='Y')
updated_dim_df.show()

+-----------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+----------+-------+------+
|       1001|Argha| Kolkata|2023-01-01|   null|     Y|
|       1002|Rahul|   Delhi|2023-01-01|   null|     Y|
|       1003| John|  Mumbai|2023-01-05|   null|     Y|
|       1004| Sham| Chennai|2023-01-08|   null|     Y|
+-----------+-----+--------+----------+-------+------+



In [7]:
join_df = src_df.alias('src').join(
    updated_dim_df.alias('dim'),
    how='left',
    on='Customer_id'
)
join_df.show()

+-----------+-----+--------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+-----+--------+----------+-------+------+
|       1001|Argha|  Mumbai|Argha| Kolkata|2023-01-01|   null|     Y|
|       1002|Rahul|   Delhi|Rahul|   Delhi|2023-01-01|   null|     Y|
|       1003| John|    pune| John|  Mumbai|2023-01-05|   null|     Y|
|       1006| Amit|    Pune| null|    null|      null|   null|  null|
+-----------+-----+--------+-----+--------+----------+-------+------+



In [8]:
changed_df = join_df.filter(
    (col('dim.Customer_id').isNotNull())&
    (col('src.Name')!=col('dim.Name'))|
    (col('src.Location')!=col('dim.Location'))
)

changed_df.show()

+-----------+-----+--------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+-----+--------+----------+-------+------+
|       1001|Argha|  Mumbai|Argha| Kolkata|2023-01-01|   null|     Y|
|       1003| John|    pune| John|  Mumbai|2023-01-05|   null|     Y|
+-----------+-----+--------+-----+--------+----------+-------+------+



In [9]:
expire_df = changed_df.select(
    col('dim.Customer_id'),
    col('dim.Name'),
    col('dim.Location'),
    col('dim.StartDate'),
    current_date().alias('EndDate'),
    lit('N').alias('Status')
)

expire_df.show()

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1001|Argha| Kolkata|2023-01-01|2026-08-04|     N|
|       1003| John|  Mumbai|2023-01-05|2026-08-04|     N|
+-----------+-----+--------+----------+----------+------+



In [10]:
insert_df = changed_df.select(
    col('src.customer_id'),
    col('src.Name'),
    col('src.Location'),
    current_date().alias("StartDate"),
    lit(None).alias('EndDate'),
    lit('Y').alias('Status')
)

insert_df.show()

+-----------+-----+--------+----------+-------+------+
|customer_id| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+----------+-------+------+
|       1001|Argha|  Mumbai|2026-08-04|   null|     Y|
|       1003| John|    pune|2026-08-04|   null|     Y|
+-----------+-----+--------+----------+-------+------+



In [11]:
new_customer = join_df.filter(
    col('dim.Customer_id').isNull()).select(
        col('src.Customer_id'),
        col('src.Name'),
        col('src.Location'),
        current_date().alias("StartDate"),
        lit(None).alias('EndDate'),
        lit('Y').alias('Status')
    )

new_customer.show()

+-----------+----+--------+----------+-------+------+
|Customer_id|Name|Location| StartDate|EndDate|Status|
+-----------+----+--------+----------+-------+------+
|       1006|Amit|    Pune|2026-08-04|   null|     Y|
+-----------+----+--------+----------+-------+------+



In [12]:
unchange_df = dim_df.join(
    changed_df.select('Customer_id'),
    how = 'left_anti',
    on='Customer_id'
)

unchange_df.show()

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1002|Rahul|   Delhi|2023-01-01|      null|     Y|
|       1004| Sham| Chennai|2023-01-08|      null|     Y|
|       1005|Champ|   Noida|2022-05-04|2025-03-02|     N|
+-----------+-----+--------+----------+----------+------+



In [13]:
final = unchange_df.unionByName(new_customer)\
                    .unionByName(expire_df)\
                    .unionByName(insert_df)

final.show()

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1002|Rahul|   Delhi|2023-01-01|      null|     Y|
|       1004| Sham| Chennai|2023-01-08|      null|     Y|
|       1005|Champ|   Noida|2022-05-04|2025-03-02|     N|
|       1006| Amit|    Pune|2026-08-04|      null|     Y|
|       1001|Argha| Kolkata|2023-01-01|2026-08-04|     N|
|       1003| John|  Mumbai|2023-01-05|2026-08-04|     N|
|       1001|Argha|  Mumbai|2026-08-04|      null|     Y|
|       1003| John|    pune|2026-08-04|      null|     Y|
+-----------+-----+--------+----------+----------+------+



In [14]:
# Slowly Change Dimension SCD1

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
from pyspark.sql.functions import col,lit,current_date,when

In [16]:
schema = StructType(
    [
        StructField("Customer_id",IntegerType(),True),
        StructField("Name",StringType(),True),
        StructField("Location",StringType(),True),
        StructField("StartDate",StringType(),True),
        StructField("EndDate",StringType(),True),
        StructField("Status",StringType(),True),
    ]
)

Target_data = [
    (1001, "Argha", "Kolkata", "2023-01-01", None, "Y"),
    (1002, "Rahul", "Delhi", "2023-01-01", None, "Y"),
    (1003, "John", "Mumbai", "2023-01-05", None, "Y"),
    (1004, "Sham", "Chennai", "2023-01-08", None, "Y"),
    (1005, "Champ", "Noida", "2022-05-04", "2025-03-02", "N")
]

define_schema = StructType([
    StructField("Customer_id",IntegerType(),True),
    StructField("Name",StringType(),True),
    StructField("Location",StringType(),True)
])

source_data = [
    (1001, "Argha", "Mumbai"),  # changed city
    (1002, "Rahul", "Delhi"),   # no change
    (1003, "John", "pune"),     # changed city
    (1006, "Amit", "Pune")      # new record
]

In [17]:
spark = SparkSession.builder.appName('SCD1').getOrCreate()

In [18]:
target_data = spark.createDataFrame(Target_data,schema)
source_data = spark.createDataFrame(source_data,define_schema)

In [19]:
active_df = target_data.filter(col('Status')=='Y')
inactive_df = target_data.filter(col('Status')=='N')

active_df.show()
inactive_df.show()

+-----------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+----------+-------+------+
|       1001|Argha| Kolkata|2023-01-01|   null|     Y|
|       1002|Rahul|   Delhi|2023-01-01|   null|     Y|
|       1003| John|  Mumbai|2023-01-05|   null|     Y|
|       1004| Sham| Chennai|2023-01-08|   null|     Y|
+-----------+-----+--------+----------+-------+------+

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1005|Champ|   Noida|2022-05-04|2025-03-02|     N|
+-----------+-----+--------+----------+----------+------+



In [20]:
join_df = source_data.alias('src').join(
    active_df.alias('tgt'),
    on = 'Customer_id',
    how = 'left'
)

join_df.show()

+-----------+-----+--------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+-----+--------+----------+-------+------+
|       1001|Argha|  Mumbai|Argha| Kolkata|2023-01-01|   null|     Y|
|       1002|Rahul|   Delhi|Rahul|   Delhi|2023-01-01|   null|     Y|
|       1003| John|    pune| John|  Mumbai|2023-01-05|   null|     Y|
|       1006| Amit|    Pune| null|    null|      null|   null|  null|
+-----------+-----+--------+-----+--------+----------+-------+------+



In [21]:
upsert_df = join_df.select(
    col('src.Customer_id'),
    col('src.Name'),
    col('src.Location'),
    current_date().alias('StartDate'),
    lit(None).alias('EndDate'),
    lit('Y').alias('Status')
)

upsert_df.show()

+-----------+-----+--------+----------+-------+------+
|Customer_id| Name|Location| StartDate|EndDate|Status|
+-----------+-----+--------+----------+-------+------+
|       1001|Argha|  Mumbai|2026-08-04|   null|     Y|
|       1002|Rahul|   Delhi|2026-08-04|   null|     Y|
|       1003| John|    pune|2026-08-04|   null|     Y|
|       1006| Amit|    Pune|2026-08-04|   null|     Y|
+-----------+-----+--------+----------+-------+------+



In [22]:
unchange_df = active_df.join(
    source_data.select('Customer_id'),
    on = 'Customer_id',
    how = 'left_anti'
)

unchange_df.show()

+-----------+----+--------+----------+-------+------+
|Customer_id|Name|Location| StartDate|EndDate|Status|
+-----------+----+--------+----------+-------+------+
|       1004|Sham| Chennai|2023-01-08|   null|     Y|
+-----------+----+--------+----------+-------+------+



In [23]:
final_data = unchange_df.unionByName(upsert_df)\
                        .unionByName(inactive_df)

final_data.show()

+-----------+-----+--------+----------+----------+------+
|Customer_id| Name|Location| StartDate|   EndDate|Status|
+-----------+-----+--------+----------+----------+------+
|       1004| Sham| Chennai|2023-01-08|      null|     Y|
|       1001|Argha|  Mumbai|2026-08-04|      null|     Y|
|       1002|Rahul|   Delhi|2026-08-04|      null|     Y|
|       1003| John|    pune|2026-08-04|      null|     Y|
|       1006| Amit|    Pune|2026-08-04|      null|     Y|
|       1005|Champ|   Noida|2022-05-04|2025-03-02|     N|
+-----------+-----+--------+----------+----------+------+



In [24]:
# Explode Fuction pyspark

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,col

In [26]:
spark = SparkSession.builder.appName('explode function').getOrCreate()

In [27]:
json = [
    {
        'id':1,
        'Name': 'Argha',
        'Address':{
            'City': 'kolkata',
            'State': 'WB'
        },
        'orders':[
            {
                'Order_id':201,
                'Order_ammount':756
            },
            {
                'Order_id':202,
                'Order_ammount':867
            }
        ]
    }
]

In [28]:
data = spark.createDataFrame(json)
data.show(truncate=False)

+------------------------------+-----+---+----------------------------------------------------------------------------------+
|Address                       |Name |id |orders                                                                            |
+------------------------------+-----+---+----------------------------------------------------------------------------------+
|{State -> WB, City -> kolkata}|Argha|1  |[{Order_id -> 201, Order_ammount -> 756}, {Order_id -> 202, Order_ammount -> 867}]|
+------------------------------+-----+---+----------------------------------------------------------------------------------+



In [29]:
data1 = data.select(
    col('id'),
    col('Name'),
    col('Address.City').alias('city'),
    col('Address.State').alias('state'),
    col('orders')
)
data1.show()

+---+-----+-------+-----+--------------------+
| id| Name|   city|state|              orders|
+---+-----+-------+-----+--------------------+
|  1|Argha|kolkata|   WB|[{Order_id -> 201...|
+---+-----+-------+-----+--------------------+



In [30]:
data2 = data1.withColumn('order', explode('orders'))
data3= data2.select(
    col('id'),
    col('Name'),
    col('city'),
    col('state'),
    col('order.Order_id').alias('orderid'),
    col('order.Order_ammount').alias('orderammount')
)

data3.show()

+---+-----+-------+-----+-------+------------+
| id| Name|   city|state|orderid|orderammount|
+---+-----+-------+-----+-------+------------+
|  1|Argha|kolkata|   WB|    201|         756|
|  1|Argha|kolkata|   WB|    202|         867|
+---+-----+-------+-----+-------+------------+



In [31]:
# AutoLoader in spark

In [32]:
# df = spark.readStream.format('cloudefiles').option('cloudefiles.format','parquet')\
#     .option('cloudefiles.schemaLocation','').option('clodefiles.schemaEvolutionMode','addNewColumns').load()

# df.writeStream.format('delta').outputMode('append').option('mergeSchema','true').option('checkpointLocation','')\
#     .option('path','').trigger().start()

In [33]:
# UDF

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,udf

In [2]:
spark = SparkSession.builder.appName('UDF').getOrCreate()

In [36]:
data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]
df = spark.createDataFrame(data, ["name", "age"])
df.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 34|
|  Bob| 45|
|Cathy| 29|
+-----+---+



In [37]:
def category_age(age):
    if age < 30:
        return "Young"
    elif age < 40:
        return "Middle-aged"
    else:
        return "Senior"

In [38]:
category_age_udf = udf(category_age,StringType())

In [39]:
df = df.withColumn('age_cat',category_age_udf(col('age')))
df.show()

+-----+---+-----------+
| name|age|    age_cat|
+-----+---+-----------+
|Alice| 34|Middle-aged|
|  Bob| 45|     Senior|
|Cathy| 29|      Young|
+-----+---+-----------+



In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col,udf

In [41]:
Spark = SparkSession.builder.appName('UDF_JOIN').getOrCreate()

In [42]:
schema1 = ['ID','Name','Salary']
schema2 = ['EMP_ID','Dept']

data1= [(101,'Arun',2000),(102,'Priyo',5000),('103','Gopal',7000)]
data2= [(101,'Ben'),(103,'Eng')]

df_salary = Spark.createDataFrame(data1,schema=schema1)
df_dept = Spark.createDataFrame(data2,schema=schema2)

df_salary.show()
df_dept.show()

+---+-----+------+
| ID| Name|Salary|
+---+-----+------+
|101| Arun|  2000|
|102|Priyo|  5000|
|103|Gopal|  7000|
+---+-----+------+

+------+----+
|EMP_ID|Dept|
+------+----+
|   101| Ben|
|   103| Eng|
+------+----+



In [43]:
def type_converter(value):
    result = str(value)
    return result

In [44]:
type_converter_udf = udf(type_converter,StringType())
df_salary = df_salary.withColumn('join_key',type_converter_udf(col('ID')))
df_dept = df_dept.withColumn('join_key',type_converter_udf(col('EMP_ID')))
df_salary.show()
df_dept.show()

+---+-----+------+--------+
| ID| Name|Salary|join_key|
+---+-----+------+--------+
|101| Arun|  2000|     101|
|102|Priyo|  5000|     102|
|103|Gopal|  7000|     103|
+---+-----+------+--------+

+------+----+--------+
|EMP_ID|Dept|join_key|
+------+----+--------+
|   101| Ben|     101|
|   103| Eng|     103|
+------+----+--------+



In [45]:
final_result = df_salary.join(
    df_dept,
    on = 'join_key',
    how = 'inner'
)
final_result.select(col('join_key'),col('Name'),col('Salary'),col('Dept')).show()


+--------+-----+------+----+
|join_key| Name|Salary|Dept|
+--------+-----+------+----+
|     101| Arun|  2000| Ben|
|     103|Gopal|  7000| Eng|
+--------+-----+------+----+



In [46]:
# md5 implementation || hash Function

In [47]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when,md5,concat_ws

In [48]:
spark = SparkSession.builder.appName('md5').getOrCreate()

In [49]:
data1 = [(1,'Arun','Arun'),(2,'Bikash','Bik'),(3,'Champa','Champa'),(4,'Lion','lion')]

df1 = spark.createDataFrame(data1,['ID','OldName','NewName'])
df1.show()

+---+-------+-------+
| ID|OldName|NewName|
+---+-------+-------+
|  1|   Arun|   Arun|
|  2| Bikash|    Bik|
|  3| Champa| Champa|
|  4|   Lion|   lion|
+---+-------+-------+



In [50]:
df2 =  df1.withColumn('Old_hash_value',md5(col('OldName'))).withColumn('New_hash_value',md5(col('NewName')))
df2.show()

+---+-------+-------+--------------------+--------------------+
| ID|OldName|NewName|      Old_hash_value|      New_hash_value|
+---+-------+-------+--------------------+--------------------+
|  1|   Arun|   Arun|5455c33e251ab225e...|5455c33e251ab225e...|
|  2| Bikash|    Bik|81cacd6897f1b2ea3...|5d33632717c690621...|
|  3| Champa| Champa|23c852856b82a069e...|23c852856b82a069e...|
|  4|   Lion|   lion|60d28e7d879c0dc48...|6b42d00c4ca6ddc33...|
+---+-------+-------+--------------------+--------------------+



In [51]:
final_data = df2.withColumn('status',when(col('Old_hash_value')==col('New_hash_value'),'MATCH').otherwise('NOT MATCH'))
final_data.show()

+---+-------+-------+--------------------+--------------------+---------+
| ID|OldName|NewName|      Old_hash_value|      New_hash_value|   status|
+---+-------+-------+--------------------+--------------------+---------+
|  1|   Arun|   Arun|5455c33e251ab225e...|5455c33e251ab225e...|    MATCH|
|  2| Bikash|    Bik|81cacd6897f1b2ea3...|5d33632717c690621...|NOT MATCH|
|  3| Champa| Champa|23c852856b82a069e...|23c852856b82a069e...|    MATCH|
|  4|   Lion|   lion|60d28e7d879c0dc48...|6b42d00c4ca6ddc33...|NOT MATCH|
+---+-------+-------+--------------------+--------------------+---------+



In [52]:
# Running Total/ Cumilitives process
# Duplicate Data Finding

In [53]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType,IntegerType,StructField,StructType
from pyspark.sql.window import Window
from pyspark.sql.functions import col,when,sum,row_number,rank,dense_rank

In [54]:
spark =  SparkSession.builder.appName('running Total').getOrCreate()
schema =  StructType([
    StructField('region',StringType(),True),
    StructField('Date',StringType(),True),
    StructField('Amount',StringType(),True)
])

data = [
    ("East", "2024-01-01", 100),
    ("East", "2024-01-02", 200),
    ("East", "2024-01-03", 150),
    ("West", "2024-01-01", 300),
    ("West", "2024-01-02", 100)
]

data  =  spark.createDataFrame(data,schema)
data.show()

+------+----------+------+
|region|      Date|Amount|
+------+----------+------+
|  East|2024-01-01|   100|
|  East|2024-01-02|   200|
|  East|2024-01-03|   150|
|  West|2024-01-01|   300|
|  West|2024-01-02|   100|
+------+----------+------+



In [55]:
Window_spec = Window.orderBy(col('Date').desc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)
running_total = data.withColumn('Total_ammount', sum(col('Amount')).over(Window_spec))
running_total.show()

+------+----------+------+-------------+
|region|      Date|Amount|Total_ammount|
+------+----------+------+-------------+
|  East|2024-01-03|   150|        150.0|
|  East|2024-01-02|   200|        350.0|
|  West|2024-01-02|   100|        450.0|
|  East|2024-01-01|   100|        550.0|
|  West|2024-01-01|   300|        850.0|
+------+----------+------+-------------+



In [56]:
spark =  SparkSession.builder.appName('drop_duplicate').getOrCreate()
schema =  StructType([
    StructField('ID',StringType(),True),
    StructField('Name',StringType(),True),
    StructField('Amount',IntegerType(),True)
])

data = [
    ("1", "Argha", 100),
    ("1", "Argha", 100),
    ("2", "Rohit", 500),
    ("3", "Bell", 300),
    ("2", "Rohit", 500)
]

data  =  spark.createDataFrame(data,schema)
data.show()

+---+-----+------+
| ID| Name|Amount|
+---+-----+------+
|  1|Argha|   100|
|  1|Argha|   100|
|  2|Rohit|   500|
|  3| Bell|   300|
|  2|Rohit|   500|
+---+-----+------+



In [57]:
Window_spec = Window.partitionBy(col('Id')).orderBy(col('Name').desc())

In [58]:
remove_duplicate = data.withColumn('rn', row_number().over(Window_spec))
remove_duplicate.show()

+---+-----+------+---+
| ID| Name|Amount| rn|
+---+-----+------+---+
|  1|Argha|   100|  1|
|  1|Argha|   100|  2|
|  2|Rohit|   500|  1|
|  2|Rohit|   500|  2|
|  3| Bell|   300|  1|
+---+-----+------+---+



In [59]:
remove_duplicate.filter(col('rn')==1).show()

+---+-----+------+---+
| ID| Name|Amount| rn|
+---+-----+------+---+
|  1|Argha|   100|  1|
|  2|Rohit|   500|  1|
|  3| Bell|   300|  1|
+---+-----+------+---+



In [60]:
# moving Average

In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col,when,avg,round

In [62]:
spark = SparkSession.builder.appName("MovingAverage4Day").getOrCreate()
data = [
    ("StoreA", "2024-01-01", 100),
    ("StoreA", "2024-01-02", 150),
    ("StoreA", "2024-01-03", 200),
    ("StoreA", "2024-01-04", 130),
    ("StoreA", "2024-01-05", 170),
    ("StoreA", "2024-01-06", 190),
    ("StoreA", "2024-01-07", 210),
    ("StoreB", "2024-01-01", 300),
    ("StoreB", "2024-01-02", 280),
    ("StoreB", "2024-01-03", 310),
    ("StoreB", "2024-01-04", 260),
    ("StoreB", "2024-01-05", 290),
    ("StoreB", "2024-01-06", 305),
    ("StoreB", "2024-01-07", 275),
]
columns = ["store", "date", "sales"]
data = spark.createDataFrame(data,columns)
data.show()

+------+----------+-----+
| store|      date|sales|
+------+----------+-----+
|StoreA|2024-01-01|  100|
|StoreA|2024-01-02|  150|
|StoreA|2024-01-03|  200|
|StoreA|2024-01-04|  130|
|StoreA|2024-01-05|  170|
|StoreA|2024-01-06|  190|
|StoreA|2024-01-07|  210|
|StoreB|2024-01-01|  300|
|StoreB|2024-01-02|  280|
|StoreB|2024-01-03|  310|
|StoreB|2024-01-04|  260|
|StoreB|2024-01-05|  290|
|StoreB|2024-01-06|  305|
|StoreB|2024-01-07|  275|
+------+----------+-----+



In [63]:
Window_sp = Window.partitionBy(col('store')).orderBy(col('date').desc()).rowsBetween(-2,0)
moving_avg = data.withColumn('avg_sales', round(avg(col('sales')).over(Window_sp),2))
moving_avg.show()

+------+----------+-----+---------+
| store|      date|sales|avg_sales|
+------+----------+-----+---------+
|StoreA|2024-01-07|  210|    210.0|
|StoreA|2024-01-06|  190|    200.0|
|StoreA|2024-01-05|  170|    190.0|
|StoreA|2024-01-04|  130|   163.33|
|StoreA|2024-01-03|  200|   166.67|
|StoreA|2024-01-02|  150|    160.0|
|StoreA|2024-01-01|  100|    150.0|
|StoreB|2024-01-07|  275|    275.0|
|StoreB|2024-01-06|  305|    290.0|
|StoreB|2024-01-05|  290|    290.0|
|StoreB|2024-01-04|  260|    285.0|
|StoreB|2024-01-03|  310|   286.67|
|StoreB|2024-01-02|  280|   283.33|
|StoreB|2024-01-01|  300|   296.67|
+------+----------+-----+---------+



In [64]:
data = {
    "A": [10, 25, 15, 40],
    "B": [5, 12, 8],
    "C": [30, 18, 45]
}

In [65]:
output = {}
def find_gretest_value(actual_data):
    for key,value in actual_data.items():
        maximum = max(value)
        output[key] = maximum
        return output

find_gretest_value(data)

{'A': 40}

In [66]:
# incremental Load if record is changes everyday

In [67]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when,col
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

In [68]:
# Read data
# builder = SparkSession.builder.appName('incremental_load') \
#         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#         .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 

# spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [69]:
# target_df = spark.read.format('csv').option('header','True').option('inferschema','True').load('TestData\\customer_increment.csv')

# source_df = spark.read.format('csv').option('header','True').option('inferschema','True').load('TestData\\customer_target.csv')

# target_df.show(2)
# source_df.show(2)

In [70]:
# final = target_df.alias('t').merge(
#     source_df.alias('s'),
#     "t.customer_id = s.customer_id"
# ).whenMatchedUpdate(
#     condition = "s.updated_date>t.updated_date",
#     set = {
#         "name" : "s.name",
#         "city" : "s.city",
#         "salary": "s.salary",
#         "updated_date" : "s.updated_date"
#     }
# ).whenNotMatchedInsert(
#     values = {
#         "customer_id" : "s.customer_id",
#         "name" : "s.name",
#         "city" : "s.city",
#         "salary": "s.salary",
#         "updated_date" : "s.updated_date"
#     }
# )

# final.show()

In [71]:
# Handle skwed data

In [72]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import array,explode,lit,floor,rand

In [73]:
spark = SparkSession.builder.appName('skewed_data').getOrCreate()

In [74]:
o_data = [
    (101,'oredr_1',200),
    (101,'order_2',300),
    (101,'order_3',400),
    (101,'oredr_4',500),
    (102,'order_5',600),
    (103,'order_6',1000)
]

o_column = ['cus_id','order','ammount']

c_data = [
    (101,'Pijush'),
    (102,'Sajol'),
    (103,'Arun')
]
c_column = ['cus_id','name']

order_df = spark.createDataFrame(o_data,o_column)
customer_df = spark.createDataFrame(c_data,c_column)
order_df.show(2)
customer_df.show(2)

+------+-------+-------+
|cus_id|  order|ammount|
+------+-------+-------+
|   101|oredr_1|    200|
|   101|order_2|    300|
+------+-------+-------+
only showing top 2 rows

+------+------+
|cus_id|  name|
+------+------+
|   101|Pijush|
|   102| Sajol|
+------+------+
only showing top 2 rows



In [75]:
salt_range = 3
salted_order_df =  order_df.withColumn('salt', floor(rand() * salt_range))
salted_order_df.show()

+------+-------+-------+----+
|cus_id|  order|ammount|salt|
+------+-------+-------+----+
|   101|oredr_1|    200|   1|
|   101|order_2|    300|   1|
|   101|order_3|    400|   1|
|   101|oredr_4|    500|   2|
|   102|order_5|    600|   2|
|   103|order_6|   1000|   0|
+------+-------+-------+----+



In [76]:
customer_expand_df = customer_df.withColumn('salt',explode(array([lit(i) for i in range(salt_range)])))
customer_expand_df.show()

+------+------+----+
|cus_id|  name|salt|
+------+------+----+
|   101|Pijush|   0|
|   101|Pijush|   1|
|   101|Pijush|   2|
|   102| Sajol|   0|
|   102| Sajol|   1|
|   102| Sajol|   2|
|   103|  Arun|   0|
|   103|  Arun|   1|
|   103|  Arun|   2|
+------+------+----+



In [77]:
join_df = salted_order_df.join(
    customer_expand_df,
    on = ['cus_id','salt']
)
final = join_df.drop('salt')
final.show()

+------+-------+-------+------+
|cus_id|  order|ammount|  name|
+------+-------+-------+------+
|   101|oredr_1|    200|Pijush|
|   101|order_2|    300|Pijush|
|   101|order_3|    400|Pijush|
|   101|oredr_4|    500|Pijush|
|   102|order_5|    600| Sajol|
|   103|order_6|   1000|  Arun|
+------+-------+-------+------+



In [78]:
# find out null values

In [79]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, when

In [80]:
spark = SparkSession.builder.appName('check_null').getOrCreate()
sales_df = spark.read.format('csv').option('inferschema','True').option('header','True').load('TestData\\sales_with_null_values.csv')
sales_df.show(2)

+--------+-------+--------+-----+--------+
|order_id|product|quantity|price|discount|
+--------+-------+--------+-----+--------+
|       1| Laptop|       2|50000|     0.1|
|       2|  Mouse|    null|  800|    0.05|
+--------+-------+--------+-----+--------+
only showing top 2 rows



In [83]:
find_null_values =  sales_df.select(
    [
        count(when(col(c).isNull(), c)).alias(c) for c in sales_df.columns
                ]
)

find_null_values.show()

+--------+-------+--------+-----+--------+
|order_id|product|quantity|price|discount|
+--------+-------+--------+-----+--------+
|       0|      0|       0|    0|       0|
+--------+-------+--------+-----+--------+



In [82]:
sales_df = sales_df.fillna(
    {
        "quantity":0,
        "price":0,
        "discount":0
    }
)

In [5]:
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.appName('read_parqet').master('local[*]').getOrCreate()
df = spark.read.parquet("streamingProject/output")
df.show()

AnalysisException: Unable to infer schema for Parquet at . It must be specified manually.

In [8]:
import os

print(os.getcwd())
print(os.path.exists("streamingProject/output"))
print(os.listdir("streamingProject/output"))

d:\ADF_DB
True
['.part-00000-9a45b73d-3917-4715-b1db-0b8307e5ef45-c000.snappy.parquet.crc', '.part-00000-b0be1ecc-1db1-43d7-8402-5677ad37dda3-c000.snappy.parquet.crc', 'part-00000-9a45b73d-3917-4715-b1db-0b8307e5ef45-c000.snappy.parquet', 'part-00000-b0be1ecc-1db1-43d7-8402-5677ad37dda3-c000.snappy.parquet', '_spark_metadata']


In [11]:
df = spark.read.parquet(
    r"D:\ADF_DB\streamingProject\output\part-00000-b0be1ecc-1db1-43d7-8402-5677ad37dda3-c000.snappy.parquet"
)

df.show()

+-----------+-----+------+
|customer_id| name|amount|
+-----------+-----+------+
|        101|Rahul|   500|
|        102| Amit|   700|
|        103|Priya|   900|
+-----------+-----+------+



In [10]:
import os

for f in os.listdir("streamingProject/output"):
    if f.endswith(".parquet"):
        path = os.path.join("streamingProject/output", f)
        print(f, os.path.getsize(path))

part-00000-9a45b73d-3917-4715-b1db-0b8307e5ef45-c000.snappy.parquet 481
part-00000-b0be1ecc-1db1-43d7-8402-5677ad37dda3-c000.snappy.parquet 967
